In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv("/kaggle/input/q3-ka-ai-2026/Q3_data.csv")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
df = df.dropna(subset=["Target"])
cate = df.select_dtypes(include="object").columns

for col in cate:
  df[col] = df[col].fillna("None")

num = df.select_dtypes(include=["int64","float64"]).columns.drop("Target")

for col in num:
  df[col] = df[col].fillna(df[col].mode()[0])


In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder
#no need all featrues in numrical

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[num] = scaler.fit_transform(df[num])
df.head()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
print(df["Target"].value_counts())
plt.figure(figsize=(10, 5))
plt.bar(df["Target"].value_counts().index,df["Target"].value_counts().values, edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('Target')
plt.ylabel('Counts')
plt.show()

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
%pip install catboost

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier
import numpy as np
n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
model = CatBoostClassifier(verbose=0,n_estimators=320, max_depth=4)
all_acc =[]
all_f1 = []
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models

  print(f"Training...")
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)

  f1 = f1_score(y_test, y_pred, zero_division=0)

  all_acc.append(accuracy)
  all_f1.append(f1)
print(f"Avg Acc = {np.mean(all_acc)}")
print(f"Avg F1-score = {np.mean(all_f1)}")

In [ ]:
# Task 1: Write your code here:
Top = 15

features = df.columns.drop("Target")
fi = pd.DataFrame({"CatBoost":model.feature_importances_},index=model.feature_names_).head(Top)
imp = fi["CatBoost"]
sorted_idx = np.argsort(imp)
plt.barh(features[sorted_idx], imp[sorted_idx])

plt.title(f"CatBoost top 15 Feature Importance")
plt.xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
golden_one = fi.sort_values("CatBoost",ascending=False)
print(f"The Golden Feature is {golden_one.index[0]} with a {golden_one.iloc[0,0]:.4f}% importance")


In [ ]:
# Task Bonus: Write your code here:
# Task 1: Write your code here:
X_new = X[golden_one.index[0]].astype(float)

skf1 = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
model2 = CatBoostClassifier(verbose=0,n_estimators=320, max_depth=4)
all_acc2 =[]
all_f12 = []
for fold_idx, (train_index, test_index) in enumerate(skf1.split(X_new, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models

  print(f"Training...")
  model2.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)

  f1 = f1_score(y_test, y_pred, zero_division=0)

  all_acc2.append(accuracy)
  all_f12.append(f1)
print(f"old Avg Acc = {np.mean(all_acc)}")
print(f"old Avg F1-score = {np.mean(all_f1)}")
print(f"new Avg Acc = {np.mean(all_acc2)}")
print(f"new Avg F1-score = {np.mean(all_f12)}")


In [ ]:
#WoW